# 🔺 Triplestore Demo — RDF + SPARQL

**Gruppe 2: Semantic Triple Stores**  ·  Tool: **Oxigraph** (Rust, kein JVM)  ·  Datensatz: Deutsche Universitäten

Dieses Notebook führt die Live-Demo Query für Query vor — direkt vorne projizierbar.
Jede Query hat eine **Erklärzelle** (Markdown) und eine **Ausführzelle** (Code mit Ergebnis-Tabelle).

---
### Ablauf
1. Oxigraph starten (Docker)
2. Setup-Zelle ausführen (Verbindung + Daten laden)
3. Queries 1–7 nacheinander zeigen

**Kern-Aha-Moment:** Query 4 (0 Ergebnisse) → Query 5 (7 Ergebnisse durch Inferencing)

## 0 · Setup

**Vorher im Terminal** (im Ordner `triplestore-demo/`):
```bash
docker-compose up -d
```
Web-UI zur Kontrolle: <http://localhost:7878>

Falls nötig einmalig die Python-Pakete installieren (nächste Zelle).

In [ ]:
# Einmalig ausführen, falls Pakete fehlen:
# %pip install requests pandas

In [28]:
import requests
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# Lange Texte (z.B. Wikidata-Beschreibungen in Query 7) nicht abschneiden
pd.set_option("display.max_colwidth", None)

ENDPOINT = "http://localhost:7878"
DATA_FILE = Path("data/universitaeten.ttl")

# Prefixe, die in JEDER lokalen Query implizit vorangestellt werden -> Zellen bleiben kurz
PREFIXES = """
PREFIX uni:  <http://example.org/uni/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX wd:   <http://www.wikidata.org/entity/>
"""

def _shorten(value: str) -> str:
    """Lange URIs für die Anzeige kürzen."""
    if value.startswith("http://example.org/uni/"):
        return "uni:" + value[len("http://example.org/uni/"):]
    if value.startswith("http://www.w3.org/2000/01/rdf-schema#"):
        return "rdfs:" + value[len("http://www.w3.org/2000/01/rdf-schema#"):]
    if value.startswith("http://www.wikidata.org/entity/"):
        return "wd:" + value[len("http://www.wikidata.org/entity/"):]
    if value.startswith("http://dbpedia.org/resource/"):
        return "dbr:" + value[len("http://dbpedia.org/resource/"):]
    return value

def sparql(query: str, show_query: bool = True) -> pd.DataFrame:
    """Query an den lokalen Oxigraph senden und als DataFrame zurückgeben."""
    full = PREFIXES + query
    if show_query:
        display(Markdown("```sparql\n" + query.strip() + "\n```"))
    r = requests.post(
        f"{ENDPOINT}/query",
        data=full.encode("utf-8"),
        headers={
            "Content-Type": "application/sparql-query; charset=utf-8",
            "Accept": "application/sparql-results+json",
        },
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    cols = data["head"]["vars"]
    rows = []
    for b in data["results"]["bindings"]:
        rows.append({c: _shorten(b[c]["value"]) if c in b else "—" for c in cols})
    df = pd.DataFrame(rows, columns=cols)
    print(f"➡  {len(df)} Ergebnis(se)")
    return df

print("Helper geladen. Endpoint:", ENDPOINT)

Helper geladen. Endpoint: http://localhost:7878


In [29]:
# Verbindung prüfen + Daten in den Default-Graph laden (PUT /store?default)
health = requests.get(f"{ENDPOINT}/query", params={"query": "SELECT * WHERE {} LIMIT 1"}, timeout=10)
assert health.status_code == 200, "Oxigraph nicht erreichbar — lief 'docker-compose up -d'?"
print("✓ Oxigraph läuft")

put = requests.put(
    f"{ENDPOINT}/store?default",
    data=DATA_FILE.read_bytes(),
    headers={"Content-Type": "text/turtle"},
    timeout=30,
)
assert 200 <= put.status_code < 300, f"Laden fehlgeschlagen: HTTP {put.status_code}"
print(f"✓ Daten geladen (HTTP {put.status_code})")

total = sparql("SELECT (COUNT(*) AS ?triples) WHERE { ?s ?p ?o }", show_query=False)
display(total)

✓ Oxigraph läuft
✓ Daten geladen (HTTP 204)
➡  1 Ergebnis(se)


,triples
0,83


## Der Datensatz

### Ontologie — das Herzstück (ermöglicht Inferencing)
```turtle
uni:Person          a rdfs:Class .
uni:AcademicPerson  rdfs:subClassOf uni:Person .
uni:Professor       rdfs:subClassOf uni:AcademicPerson .
uni:Student         rdfs:subClassOf uni:AcademicPerson .
```

### Instanzen
| Typ | Einträge |
|---|---|
| 4 Städte | Stuttgart, Karlsruhe, München, Berlin |
| 4 Unis | UniStuttgart, KIT, TUMünchen, FUBerlin |
| 3 Professoren | Müller (Stuttgart), Schneider (KIT), Weber (TUM) |
| 4 Studenten | Alice + Bob (Stuttgart), Carla (KIT), David (TUM) |

**Wichtig:** Niemand ist explizit als `uni:Person` eingetragen — alle sind `uni:Professor` oder `uni:Student`. Das ist das Setup für den Aha-Moment.

---
## Query 1 — Alle Universitäten
Einfachste Form: **zwei Triple-Muster**, verbunden über dieselbe Variable `?u`.
Erst Typ suchen, dann den Namen derselben Ressource lesen. **Erwartung: 4 Zeilen.**

In [30]:
sparql("""
SELECT ?u ?name
WHERE {
    ?u a uni:University .
    ?u rdfs:label ?name .
}
ORDER BY ?name
""")

```sparql
SELECT ?u ?name
WHERE {
    ?u a uni:University .
    ?u rdfs:label ?name .
}
ORDER BY ?name
```

➡  4 Ergebnis(se)


,u,name
0,uni:FUBerlin,Freie Universität Berlin
1,uni:KIT,Karlsruher Institut für Technologie
2,uni:TUMuenchen,Technische Universität München
3,uni:UniStuttgart,Universität Stuttgart


---
## Query 2 — Unis in Baden-Württemberg
Fünf Muster, die **gleichzeitig** erfüllt sein müssen. `?u` und `?stadt` sind die Verknüpfungsvariablen.
In SQL wäre das ein JOIN über zwei Tabellen — hier einfach ein weiteres Triple-Muster, **kein `JOIN`, kein `ON`.**


In [31]:
sparql("""
SELECT ?uniname ?stadtname
WHERE {
    ?u     a            uni:University .
    ?u     rdfs:label   ?uniname .
    ?u     uni:location ?stadt .
    ?stadt rdfs:label   ?stadtname .
    ?stadt uni:bundesland "Baden-Württemberg" .
}
ORDER BY ?uniname
""")

```sparql
SELECT ?uniname ?stadtname
WHERE {
    ?u     a            uni:University .
    ?u     rdfs:label   ?uniname .
    ?u     uni:location ?stadt .
    ?stadt rdfs:label   ?stadtname .
    ?stadt uni:bundesland "Baden-Württemberg" .
}
ORDER BY ?uniname
```

➡  2 Ergebnis(se)


,uniname,stadtname
0,Karlsruher Institut für Technologie,Karlsruhe
1,Universität Stuttgart,Stuttgart


---
## Query 3 — Studenten an der Uni Stuttgart
`uni:UniStuttgart` ist hier **keine Variable, sondern eine konkrete URI** → wirkt als harter Filter.

⚠️`?s a uni:Student` muss dabei sein. Ohne diese Zeile erschienen auch Professoren. Der Variablenname `?s` sagt dem Store *nichts* über den Typ!

In [32]:
sparql("""
SELECT ?name ?fach ?semester
WHERE {
    ?s a               uni:Student .
    ?s rdfs:label      ?name .
    ?s uni:studiesAt   uni:UniStuttgart .
    ?s uni:studienfach ?fach .
    ?s uni:semester    ?semester .
}
ORDER BY ?semester
""")

```sparql
SELECT ?name ?fach ?semester
WHERE {
    ?s a               uni:Student .
    ?s rdfs:label      ?name .
    ?s uni:studiesAt   uni:UniStuttgart .
    ?s uni:studienfach ?fach .
    ?s uni:semester    ?semester .
}
ORDER BY ?semester
```

➡  2 Ergebnis(se)


,name,fach,semester
0,Bob Fischer,Elektrotechnik,3
1,Alice Schmidt,Informatik,5


---
## 🔴 Query 4 — Alle Personen, OHNE Inferencing 

> **Frage ans Publikum:** Wie viele Ergebnisse erwartet ihr?
> (Es gibt 7 Personen im Datensatz: 3 Professoren + 4 Studenten.)

Der Store sucht Triples der Form `?person rdf:type uni:Person`. Im Datensatz existiert **kein einziges** solches Triple — alle sind `uni:Professor` oder `uni:Student`.


In [33]:
sparql("""
SELECT ?person ?name
WHERE {
    ?person a          uni:Person .
    ?person rdfs:label ?name .
}
""")

```sparql
SELECT ?person ?name
WHERE {
    ?person a          uni:Person .
    ?person rdfs:label ?name .
}
```

➡  0 Ergebnis(se)


,person,name


---
## 🟢 Query 5 — Alle Personen, MIT Inferencing  

Lösung: der SPARQL **Property Path** `rdfs:subClassOf*`. Das `*` heißt *"null oder mehr Schritte"*.
Der Store traversiert die Klassen-Hierarchie:
```
uni:Professor → rdfs:subClassOf → uni:AcademicPerson → rdfs:subClassOf → uni:Person ✓
uni:Student   → rdfs:subClassOf → uni:AcademicPerson → rdfs:subClassOf → uni:Person ✓
```
Die Ontologie-Regel wurde **einmal** definiert und gilt automatisch für alle Einträge. In SQL müsste man das bei jedem INSERT manuell pflegen. 

In [34]:
sparql("""
SELECT ?person ?name ?typ
WHERE {
    ?person a          ?typ .
    ?typ    rdfs:subClassOf* uni:Person .
    ?person rdfs:label ?name .
}
ORDER BY ?typ ?name
""")

```sparql
SELECT ?person ?name ?typ
WHERE {
    ?person a          ?typ .
    ?typ    rdfs:subClassOf* uni:Person .
    ?person rdfs:label ?name .
}
ORDER BY ?typ ?name
```

➡  7 Ergebnis(se)


,person,name,typ
0,uni:ProfMueller,Prof. Dr. Müller,uni:Professor
1,uni:ProfSchneider,Prof. Dr. Schneider,uni:Professor
2,uni:ProfWeber,Prof. Dr. Weber,uni:Professor
3,uni:AliceSchmidt,Alice Schmidt,uni:Student
4,uni:BobFischer,Bob Fischer,uni:Student
5,uni:CarlaHoffmann,Carla Hoffmann,uni:Student
6,uni:DavidKoch,David Koch,uni:Student


---
## Query 6 — COUNT: Personen pro Universität
Neu hier:
- `(uni:studiesAt | uni:worksAt)` — **Alternativpfad**: Studenten *studieren*, Professoren *arbeiten* — beides in einem Schritt.
- `COUNT(...) AS ?personenanzahl` + `GROUP BY` — Aggregation **genau wie in SQL**.
- Nutzt weiterhin `rdfs:subClassOf*` aus Query 5.


In [35]:
sparql("""
SELECT ?uniname (COUNT(?person) AS ?personenanzahl)
WHERE {
    ?person a          ?typ .
    ?typ    rdfs:subClassOf* uni:Person .
    ?person (uni:studiesAt | uni:worksAt) ?u .
    ?u      rdfs:label ?uniname .
}
GROUP BY ?uniname
ORDER BY DESC(?personenanzahl)
""")

```sparql
SELECT ?uniname (COUNT(?person) AS ?personenanzahl)
WHERE {
    ?person a          ?typ .
    ?typ    rdfs:subClassOf* uni:Person .
    ?person (uni:studiesAt | uni:worksAt) ?u .
    ?u      rdfs:label ?uniname .
}
GROUP BY ?uniname
ORDER BY DESC(?personenanzahl)
```

➡  3 Ergebnis(se)


,uniname,personenanzahl
0,Universität Stuttgart,3
1,Technische Universität München,2
2,Karlsruher Institut für Technologie,2


---
## Query 7 — Linked Data: lokal × Wikidata 

**Idee:** Zwei unserer Unis verlinken ihren Standort **direkt** auf eine globale Wikidata-Ressource (`uni:TUMuenchen uni:location wd:Q1726`, `uni:FUBerlin uni:location wd:Q64`) — statt auf einen lokalen `uni:City`-Knoten. S
tuttgart + Karlsruhe bleiben rein lokal (zum Vergleich). 
Über diese gemeinsame, globale URI reichern wir die lokalen Daten live mit Wikidata an (Label + deutsche Kurzbeschreibung).

> 💡 *"Wir müssen nicht alle Daten der Welt selbst speichern — wir verlinken einfach auf andere Wissensgraphen."*

**Eine** föderierte Query über `SERVICE` — der lokale Oxigraph schickt den `SERVICE`-Block live an Wikidata und führt die Ergebnisse zusammen. 


⚠️ Braucht Internet.

In [37]:
sparql("""
PREFIX schema: <http://schema.org/>
PREFIX wd:     <http://www.wikidata.org/entity/>

SELECT ?uniname ?stadtname ?description
WHERE {
    # Aus unserem lokalen Store: Unis, deren Standort direkt auf Wikidata zeigt
    ?u  a uni:University .
    ?u  rdfs:label ?uniname .
    ?u  uni:location ?wdCity .
    FILTER (STRSTARTS(STR(?wdCity), "http://www.wikidata.org/"))

    # Von Wikidata: Label + Beschreibung der verlinkten Stadt-Ressource.
    SERVICE <https://query.wikidata.org/sparql> {
        VALUES ?wdCity { wd:Q1726 wd:Q64 }
        ?wdCity schema:description ?description .
        FILTER (lang(?description) = "de")
        ?wdCity rdfs:label ?stadtname .
        FILTER (lang(?stadtname) = "de")
    }
}
ORDER BY ?uniname
""")

```sparql
PREFIX schema: <http://schema.org/>
PREFIX wd:     <http://www.wikidata.org/entity/>

SELECT ?uniname ?stadtname ?description
WHERE {
    # Aus unserem lokalen Store: Unis, deren Standort direkt auf Wikidata zeigt
    ?u  a uni:University .
    ?u  rdfs:label ?uniname .
    ?u  uni:location ?wdCity .
    FILTER (STRSTARTS(STR(?wdCity), "http://www.wikidata.org/"))

    # Von Wikidata: Label + Beschreibung der verlinkten Stadt-Ressource.
    SERVICE <https://query.wikidata.org/sparql> {
        VALUES ?wdCity { wd:Q1726 wd:Q64 }
        ?wdCity schema:description ?description .
        FILTER (lang(?description) = "de")
        ?wdCity rdfs:label ?stadtname .
        FILTER (lang(?stadtname) = "de")
    }
}
ORDER BY ?uniname
```

➡  2 Ergebnis(se)


,uniname,stadtname,description
0,Freie Universität Berlin,Berlin,Hauptstadt und bevölkerungsreichste Stadt der Bundesrepublik Deutschland
1,Technische Universität München,München,"Landeshauptstadt des Freistaates Bayern, Deutschland"
